In [9]:
import numpy as np
import glob
import os

# --- Get one of your input images ---
VAL_DIR = "data/UDC-SIT_subset/val"
input_paths = sorted(glob.glob(os.path.join(VAL_DIR, "input", "*.npy")))

if not input_paths:
    print(f"Error: No .npy files found in {os.path.join(VAL_DIR, 'input')}")
else:
    file_path = input_paths[0]
    
    try:
        data = np.load(file_path)
        
        print(f"--- Metadata for: {os.path.basename(file_path)} ---")
        print(f"Data Type (dtype): {data.dtype}")
        print(f"Data Shape: {data.shape}")
        
        # Check the value range
        print(f"Min value: {data.min()}")
        print(f"Max value: {data.max()}")
        print(f"Mean value: {data.mean()}")
        
    except Exception as e:
        print(f"An error occurred: {e}")

--- Metadata for: 1004.npy ---
Data Type (dtype): float32
Data Shape: (1792, 1280, 4)
Min value: 0.0
Max value: 1023.0
Mean value: 129.1371307373047


In [10]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from models.mambair_teacher import FrequencyAwareTeacher

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# --- LOAD THE NEW 10-BIT MODEL ---
MODEL_WEIGHTS_PATH = "teacher_10bit_normalized.pth" 
VAL_DIR = "data/UDC-SIT_subset/val"
PATCH_SIZE = 256
# ---------------------

def load_and_process_patch(npy_path, device, patch_size=256):
    """Loads a .npy file, normalizes it, and extracts a center patch."""
    img = np.load(npy_path) # Data is [0, 1023]
    
    # Get a center crop
    H, W, _ = img.shape
    ps = patch_size
    ps_h = min(ps, H)
    ps_w = min(ps, W)
    rr = (H - ps_h) // 2
    rc = (W - ps_w) // 2
    patch = img[rr : rr + ps_h, rc : rc + ps_w, :]

    # Convert patch to tensor
    img_tensor = torch.from_numpy(patch.copy()).permute(2, 0, 1).float()
    
    # --- NORMALIZE THE INPUT ---
    img_tensor /= 1023.0 # Use the correct 10-bit max value
    
    # Add batch dimension
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    # Return tensor and the NORMALIZED patch for display
    return img_tensor, patch / 1023.0 

def post_process_output(output_tensor):
    """Converts model output tensor (already in [0, 1] range) to numpy."""
    img = output_tensor.squeeze(0).detach().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    # The new model outputs [0, 1], so we just clip
    img = np.clip(img, 0, 1)
    return img

# 1. Load the model
print(f"Loading model from {MODEL_WEIGHTS_PATH}...")
model = FrequencyAwareTeacher(in_channels=4, out_channels=3).to(DEVICE)
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH))
model.eval()
print("Model loaded.")

# 2. Get one validation image
input_paths = sorted(glob.glob(os.path.join(VAL_DIR, "input", "*.npy")))
if not input_paths:
    print(f"Error: No .npy files found in {os.path.join(VAL_DIR, 'input')}")
else:
    input_path = input_paths[0]
    gt_path = input_path.replace("/input/", "/GT/")
    
    # 3. Load and preprocess patches
    udc_tensor, udc_patch_np = load_and_process_patch(input_path, DEVICE, PATCH_SIZE)
    _, gt_patch_np = load_and_process_patch(gt_path, DEVICE, PATCH_SIZE)
    
    print(f"Processing center patch of: {os.path.basename(input_path)}")
    
    # 4. Run inference
    with torch.no_grad():
        output_tensor = model(udc_tensor)
        
    # 5. Post-process the output
    restored_img__np = post_process_output(output_tensor)

    # 6. Visualize the results
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    
    axes[0].imshow(udc_patch_np[:, :, :3])
    axes[0].set_title("Input Patch (UDC)")
    axes[0].axis("off")
    
    axes[1].imshow(restored_img_np)
    axes[1].set_title("Model Output (Restored)")
    axes[1].axis("off")
    
    axes[2].imshow(gt_patch_np[:, :, :3])
    axes[2].set_title("Ground Truth Patch")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()

ModuleNotFoundError: No module named 'basicsr'

In [ ]:
# File: viz_comparisons_srgb.py
"""
Visualize sRGB comparisons for UDC-SIT:

For each image, create a 1x4 panel:
    [ Input | Teacher | Student | Ground Truth ]

Assumes:
- Ground truth dataset under DATA_ROOT with structure:
    DATA_ROOT/
        validation/input/*.npy
        validation/GT/*.npy
        test/input/*.npy
        test/GT/*.npy

- Predictions saved by testing_full.py under:
    RESULTS_ROOT/teacher_full_<split>/npy/*.npy
    RESULTS_ROOT/student_full_<split>/npy/*.npy

Each .npy file is (H, W, 4) with channels (GR, R, B, GB),
values either in [0,1023] or [0,1].

Outputs:
- Visualization PNGs under:
    VIZ_ROOT/<split>/*_panel.png

Optionally:
- Copies the PNGs to:
    DRIVE_VIZ_ROOT/<split>/*.png
"""

import os
from glob import glob
import shutil

import numpy as np
import matplotlib.pyplot as plt

# ------------- CONFIG -------------

# ⚠️ Set this to your actual dataset root
# Example (your case if testing_full.py used this):
#   DATA_ROOT = "/content/dataset/UDC-SIT"
DATA_ROOT = "data/UDC-SIT_full/UDC-SIT"

# Where testing_full.py predictions live
RESULTS_ROOT = "results_full"

# Where to save visualization PNGs locally
VIZ_ROOT = "viz_full_srgb"

# Where to copy PNGs on Google Drive (optional)
DRIVE_VIZ_ROOT = "/content/drive/MyDrive/Computational Imaging Project/viz_full_srgb"

# Splits to visualize (must match your dataset folder names)
# If your folders are "validation" and "test", keep this.
SPLITS = ["validation", "test"]

# Maximum number of images per split (None = all)
MAX_IMAGES_PER_SPLIT = 50  # or None for all

# If True, also save separate sRGB images for each component
SAVE_INDIVIDUAL_IMAGES = False

# ----------------------------------


def bayer4_to_rgb_np(arr_4ch,
                     r_gain: float = 1.9,
                     b_gain: float = 1.9,
                     apply_gamma: bool = True) -> np.ndarray:
    """
    Convert a 4-channel UDC-SIT Bayer-like image to an sRGB-like RGB image.

    Args:
        arr_4ch: (H, W, 4) np.ndarray in [0,1023] or [0,1],
                 channels: (GR, R, B, GB).
        r_gain, b_gain: white-balance gains.
        apply_gamma: if True, apply a simple gamma (~2.2).

    Returns:
        rgb: (H, W, 3) float32 in [0,1].
    """
    arr = np.asarray(arr_4ch, dtype=np.float32)
    if arr.ndim != 3 or arr.shape[-1] != 4:
        raise ValueError(f"Expected (H,W,4), got {arr.shape}")

    GR = arr[..., 0]
    R  = arr[..., 1]
    B  = arr[..., 2]
    GB = arr[..., 3]

    # Decide if we are in [0,1023] or already [0,1]
    scale = 1023.0 if arr.max() > 2.0 else 1.0

    G  = (GR + GB) / (2.0 * scale)
    Rn = (R / scale) * r_gain
    Bn = (B / scale) * b_gain

    rgb = np.stack([Rn, G, Bn], axis=-1)
    rgb = np.clip(rgb, 0.0, 1.0)

    if apply_gamma:
        rgb = np.power(rgb, 1.0 / 2.2)

    return rgb.astype(np.float32)


def load_4ch_npy(path: str) -> np.ndarray:
    """
    Load a 4-channel .npy and sanity check shape.
    """
    arr = np.load(path)
    if arr.ndim != 3 or arr.shape[-1] != 4:
        raise ValueError(f"[load_4ch_npy] {path} has shape {arr.shape}, expected (H,W,4)")
    return arr


def save_panel_png(out_path: str,
                   inp_srgb: np.ndarray,
                   teacher_srgb: np.ndarray | None,
                   student_srgb: np.ndarray | None,
                   gt_srgb: np.ndarray,
                   title: str):
    """
    Create and save a 1x4 panel PNG:
        [Input | Teacher | Student | GT]
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    panels = [
        ("Input",  inp_srgb),
        ("Teacher" if teacher_srgb is not None else "Teacher (missing)", teacher_srgb),
        ("Student" if student_srgb is not None else "Student (missing)", student_srgb),
        ("GT", gt_srgb),
    ]

    for ax, (label, img) in zip(axes, panels):
        ax.set_title(label, fontsize=10)
        ax.axis("off")
        if img is not None:
            ax.imshow(np.clip(img, 0.0, 1.0))
        else:
            ax.set_facecolor("black")
            ax.text(0.5, 0.5, "N/A", color="white",
                    ha="center", va="center", fontsize=12)

    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=[0, 0.0, 1, 0.95])

    fig.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)


def save_single_png(out_path: str, img: np.ndarray, title: str = ""):
    """
    Save a single sRGB image as PNG for debugging.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)
    ax.imshow(np.clip(img, 0.0, 1.0))
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)


def process_split(split: str):
    print(f"\n=== [viz_comparisons_srgb] Split: {split} ===")

    input_dir   = os.path.join(DATA_ROOT, split, "input")
    gt_dir      = os.path.join(DATA_ROOT, split, "GT")
    teacher_dir = os.path.join(RESULTS_ROOT, f"teacher_full_{split}", "npy")
    student_dir = os.path.join(RESULTS_ROOT, f"student_full_val", "npy")

    assert os.path.isdir(input_dir), f"Missing input dir: {input_dir}"
    assert os.path.isdir(gt_dir),    f"Missing GT dir: {gt_dir}"
    if not os.path.isdir(teacher_dir):
        print(f"[viz] WARNING: teacher predictions dir not found: {teacher_dir}")
    if not os.path.isdir(student_dir):
        print(f"[viz] WARNING: student predictions dir not found: {student_dir}")

    input_files = sorted(glob(os.path.join(input_dir, "*.npy")))
    if MAX_IMAGES_PER_SPLIT is not None:
        input_files = input_files[:MAX_IMAGES_PER_SPLIT]

    n_files = len(input_files)
    print(f"[viz] Found {n_files} input images in {input_dir}")

    split_viz_dir = os.path.join(VIZ_ROOT, split)
    os.makedirs(split_viz_dir, exist_ok=True)

    for idx, inp_path in enumerate(input_files):
        base = os.path.basename(inp_path)
        name = os.path.splitext(base)[0]
        print(f"[viz] [{split}] {idx+1}/{n_files}: {base}")

        gt_path      = os.path.join(gt_dir, base)
        teacher_path = os.path.join(teacher_dir, base) if os.path.isdir(teacher_dir) else None
        student_path = os.path.join(student_dir, base) if os.path.isdir(student_dir) else None

        if not os.path.exists(gt_path):
            print(f"[viz] WARNING: GT not found for {base}, skipping.")
            continue

        # Load 4-channel arrays
        inp_4ch = load_4ch_npy(inp_path)
        gt_4ch  = load_4ch_npy(gt_path)

        teacher_4ch = load_4ch_npy(teacher_path) if teacher_path and os.path.exists(teacher_path) else None
        student_4ch = load_4ch_npy(student_path) if student_path and os.path.exists(student_path) else None

        # Convert to sRGB
        inp_srgb = bayer4_to_rgb_np(inp_4ch)
        gt_srgb  = bayer4_to_rgb_np(gt_4ch)
        teacher_srgb = bayer4_to_rgb_np(teacher_4ch) if teacher_4ch is not None else None
        student_srgb = bayer4_to_rgb_np(student_4ch) if student_4ch is not None else None

        # Save combined panel
        panel_title = f"{split} - {name}"
        panel_path  = os.path.join(split_viz_dir, f"{name}_panel.png")
        save_panel_png(panel_path, inp_srgb, teacher_srgb, student_srgb, gt_srgb, panel_title)

        # Optional per-image PNGs
        if SAVE_INDIVIDUAL_IMAGES:
            save_single_png(os.path.join(split_viz_dir, f"{name}_input.png"),   inp_srgb,  "Input")
            if teacher_srgb is not None:
                save_single_png(os.path.join(split_viz_dir, f"{name}_teacher.png"), teacher_srgb, "Teacher")
            if student_srgb is not None:
                save_single_png(os.path.join(split_viz_dir, f"{name}_student.png"), student_srgb, "Student")
            save_single_png(os.path.join(split_viz_dir, f"{name}_gt.png"),      gt_srgb,   "GT")

    # Copy to Drive if mounted
    if os.path.isdir("/content/drive"):
        drive_split_dir = os.path.join(DRIVE_VIZ_ROOT, split)
        os.makedirs(drive_split_dir, exist_ok=True)

        panel_files = glob(os.path.join(split_viz_dir, "*_panel.png"))
        for p in panel_files:
            shutil.copy(p, os.path.join(drive_split_dir, os.path.basename(p)))
        print(f"[viz] Copied {len(panel_files)} panels to Drive: {drive_split_dir}")
    else:
        print("[viz] /content/drive not found; skipping Drive copy.")


def main():
    os.makedirs(VIZ_ROOT, exist_ok=True)
    for split in SPLITS:
        process_split(split)


if __name__ == "__main__":
    main()



=== [viz_comparisons_srgb] Split: validation ===
[viz] WARNING: teacher predictions dir not found: results_full/teacher_full_validation/npy
[viz] Found 50 input images in data/UDC-SIT_full/UDC-SIT/validation/input
[viz] [validation] 1/50: 1004.npy
[viz] [validation] 2/50: 1027.npy
[viz] [validation] 3/50: 1034.npy
[viz] [validation] 4/50: 1081.npy
[viz] [validation] 5/50: 1127.npy
[viz] [validation] 6/50: 1132.npy
[viz] [validation] 7/50: 114.npy
[viz] [validation] 8/50: 1155.npy
[viz] [validation] 9/50: 1179.npy
[viz] [validation] 10/50: 1186.npy
[viz] [validation] 11/50: 1188.npy
[viz] [validation] 12/50: 1193.npy
[viz] [validation] 13/50: 1195.npy
[viz] [validation] 14/50: 1239.npy
[viz] [validation] 15/50: 1240.npy
[viz] [validation] 16/50: 1242.npy
[viz] [validation] 17/50: 1248.npy
[viz] [validation] 18/50: 1249.npy
[viz] [validation] 19/50: 1255.npy
[viz] [validation] 20/50: 1257.npy
[viz] [validation] 21/50: 1273.npy
[viz] [validation] 22/50: 1312.npy
[viz] [validation] 23/50: